# Article-style pipeline — hotel reviews (`balanced-reviews.txt`)

Same flow as `article_repro_pipeline.ipynb`, but for the **second dataset** (tab-separated hotel reviews):

1. Star ratings → binary labels (4–5 → POS, 1–2 → NEG; **3★ dropped** by default)
2. Review text cleanup via `src.preprocessing.preprocess_balanced_review`
3. **Balanced** POS/NEG sample (min class count)
4. TF-IDF + classical ML, then CNN + BiLSTM
5. Export `scripts/gridsearch_balanced_reviews.py` (GridSearchCV, runnable from repo root)


In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    accuracy_score,
)
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from preprocessing import preprocess_balanced_review, star_rating_to_binary


In [ ]:
# --- Load balanced-reviews.tsv and build binary balanced dataset ---

data_path = ROOT / "data" / "balanced-reviews.txt"
if not data_path.exists():
    raise FileNotFoundError(f"Missing dataset: {data_path}")

raw = pd.read_csv(data_path, sep="\t", encoding="utf-16")
# normalize column names (strip spaces from header cells)
raw.columns = [str(c).strip() for c in raw.columns]
need = {"review", "rating"}
missing = need - set(raw.columns)
if missing:
    raise KeyError(f"CSV missing columns {missing}; have {list(raw.columns)}")

df = raw[["review", "rating"]].copy()
df["review"] = df["review"].astype(str)
df["label"] = df["rating"].map(lambda r: star_rating_to_binary(r, neutral="drop"))
before = len(df)
df = df[df["label"].notna()].copy()
dropped = before - len(df)
print("Rows after dropping neutral (3★) / invalid ratings:", len(df), "| dropped:", dropped)
print("Label counts:", df["label"].value_counts().to_dict())

bin_df = df[df["label"].isin(["POS", "NEG"])].copy()
vc = bin_df["label"].value_counts()
min_n = int(vc.min())
pos = bin_df[bin_df["label"] == "POS"].sample(min_n, random_state=SEED)
neg = bin_df[bin_df["label"] == "NEG"].sample(min_n, random_state=SEED)
bin_df = (
    pd.concat([pos, neg], ignore_index=True)
    .sample(frac=1.0, random_state=SEED)
    .reset_index(drop=True)
)

bin_df["clean_text"] = bin_df["review"].apply(preprocess_balanced_review)
bin_df = bin_df[bin_df["clean_text"].str.strip().ne("")].reset_index(drop=True)

print("Binary balanced size:", len(bin_df))
print(bin_df["label"].value_counts().to_dict())
bin_df.rename(columns={"review": "text"}, inplace=True)
bin_df.head(3)


In [ ]:
# quick transformation examples
sample = bin_df.sample(3, random_state=SEED)[["text", "clean_text", "label"]]
sample


In [ ]:
# train/test split
X = bin_df["clean_text"].astype(str).tolist()
y = (bin_df["label"] == "POS").astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

len(X_train), len(X_test)


In [ ]:
# --- ML training ---
vectorizer = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=30000,
    sublinear_tf=True,
    norm="l2",
)

Xtr_tfidf = vectorizer.fit_transform(X_train)
Xte_tfidf = vectorizer.transform(X_test)

ml_models = {
    "LinearSVC": LinearSVC(C=1.0, max_iter=10000, dual=False),
    "LogReg": LogisticRegression(max_iter=5000),
    "MNB": MultinomialNB(),
}

ml_rows = []
trained_ml = {}
for name, model in ml_models.items():
    model.fit(Xtr_tfidf, y_train)
    pred = model.predict(Xte_tfidf)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    ml_rows.append({"model": name, "accuracy": acc, "f1": f1})
    trained_ml[name] = model

ml_results = pd.DataFrame(ml_rows).sort_values("f1", ascending=False).reset_index(drop=True)
ml_results


In [ ]:
# --- ML visualization ---
plt.figure(figsize=(7, 4))
plt.bar(ml_results["model"], ml_results["f1"], color=["#4e79a7", "#f28e2b", "#59a14f"])
plt.ylim(0, 1)
plt.title("ML F1 Comparison (balanced reviews)")
plt.ylabel("F1")
plt.show()

best_ml_name = ml_results.iloc[0]["model"]
best_ml = trained_ml[best_ml_name]
best_pred = best_ml.predict(Xte_tfidf)

print(f"Best ML model: {best_ml_name}")
print(classification_report(y_test, best_pred, digits=4))

cm = confusion_matrix(y_test, best_pred)
ConfusionMatrixDisplay(cm, display_labels=["NEG", "POS"]).plot(cmap="Blues")
plt.title(f"Confusion Matrix - {best_ml_name}")
plt.show()


In [ ]:
# --- Deep Learning (CNN + BiLSTM, Keras) ---
# If tensorflow is not installed in this kernel, install/select the right environment first.

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Conv1D,
    GlobalMaxPooling1D,
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
)

MAX_WORDS = 30000
MAX_LEN = 200  # reviews are longer than tweets
EMB_DIM = 128

tok = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tok.fit_on_texts(X_train)

Xtr_seq = pad_sequences(
    tok.texts_to_sequences(X_train), maxlen=MAX_LEN, padding="post", truncating="post"
)
Xte_seq = pad_sequences(
    tok.texts_to_sequences(X_test), maxlen=MAX_LEN, padding="post", truncating="post"
)

cnn = Sequential(
    [
        Embedding(MAX_WORDS, EMB_DIM, input_length=MAX_LEN),
        Conv1D(128, 5, activation="relu"),
        GlobalMaxPooling1D(),
        Dropout(0.3),
        Dense(1, activation="sigmoid"),
    ]
)
cnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

bilstm = Sequential(
    [
        Embedding(MAX_WORDS, EMB_DIM, input_length=MAX_LEN),
        Bidirectional(LSTM(64)),
        Dropout(0.3),
        Dense(1, activation="sigmoid"),
    ]
)
bilstm.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

hist_cnn = cnn.fit(
    Xtr_seq, y_train, validation_split=0.1, epochs=5, batch_size=64, verbose=1
)
hist_bi = bilstm.fit(
    Xtr_seq, y_train, validation_split=0.1, epochs=5, batch_size=64, verbose=1
)

cnn_pred = (cnn.predict(Xte_seq, verbose=0).ravel() >= 0.5).astype(int)
bi_pred = (bilstm.predict(Xte_seq, verbose=0).ravel() >= 0.5).astype(int)

dl_results = pd.DataFrame(
    [
        {
            "model": "CNN",
            "accuracy": accuracy_score(y_test, cnn_pred),
            "f1": f1_score(y_test, cnn_pred),
        },
        {
            "model": "BiLSTM",
            "accuracy": accuracy_score(y_test, bi_pred),
            "f1": f1_score(y_test, bi_pred),
        },
    ]
)
dl_results


In [ ]:
# --- Deep learning visualization ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_cnn.history["val_accuracy"], label="CNN")
axes[0].plot(hist_bi.history["val_accuracy"], label="BiLSTM")
axes[0].set_title("Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(hist_cnn.history["val_loss"], label="CNN")
axes[1].plot(hist_bi.history["val_loss"], label="BiLSTM")
axes[1].set_title("Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.show()

print("CNN report")
print(classification_report(y_test, cnn_pred, digits=4))
print("BiLSTM report")
print(classification_report(y_test, bi_pred, digits=4))


In [ ]:
# --- Final comparison (ML + DL) ---
all_results = pd.concat(
    [ml_results[["model", "accuracy", "f1"]], dl_results[["model", "accuracy", "f1"]]],
    ignore_index=True,
).sort_values("f1", ascending=False)

all_results


In [ ]:
# --- Export GridSearchCV script for balanced reviews ---
script = '"""GridSearchCV on balanced hotel reviews (binary stars → sentiment). Run from repo root: python scripts/gridsearch_balanced_reviews.py"""\nfrom __future__ import annotations\n\nimport random\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split, GridSearchCV\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.svm import LinearSVC\nfrom sklearn.metrics import classification_report, accuracy_score\n\nSEED = 42\nrandom.seed(SEED)\n\nROOT = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(ROOT / "src"))\nfrom preprocessing import preprocess_balanced_review, star_rating_to_binary\n\n\ndef load_xy():\n    p = ROOT / "data" / "balanced-reviews.txt"\n    raw = pd.read_csv(p, sep="\t", encoding="utf-16")\n    raw.columns = [str(c).strip() for c in raw.columns]\n    df = raw[["review", "rating"]].copy()\n    df["review"] = df["review"].astype(str)\n    df["lab"] = df["rating"].map(lambda r: star_rating_to_binary(r, neutral="drop"))\n    df = df[df["lab"].notna() & df["lab"].isin(["POS", "NEG"])].copy()\n    vc = df["lab"].value_counts()\n    min_n = int(vc.min())\n    pos = df[df["lab"] == "POS"].sample(min_n, random_state=SEED)\n    neg = df[df["lab"] == "NEG"].sample(min_n, random_state=SEED)\n    bal = pd.concat([pos, neg], ignore_index=True).sample(frac=1.0, random_state=SEED)\n    x = bal["review"].map(preprocess_balanced_review)\n    y = (bal["lab"] == "POS").astype(int)\n    m = x.str.strip().ne("")\n    return x[m], y[m]\n\n\ndef main() -> None:\n    X, y = load_xy()\n    X_train, X_test, y_train, y_test = train_test_split(\n        X, y, test_size=0.2, random_state=SEED, stratify=y\n    )\n\n    pipe = Pipeline(\n        [\n            (\n                "tfidf",\n                TfidfVectorizer(\n                    lowercase=False,\n                    ngram_range=(1, 2),\n                    min_df=2,\n                    max_df=0.95,\n                    sublinear_tf=True,\n                    norm="l2",\n                ),\n            ),\n            ("clf", LinearSVC(max_iter=10000, dual=False)),\n        ]\n    )\n\n    grid = GridSearchCV(\n        pipe,\n        {"clf__C": [0.1, 0.5, 1, 10, 100]},\n        cv=5,\n        scoring="f1",\n        n_jobs=-1,\n        verbose=1,\n    )\n    grid.fit(X_train, y_train)\n    pred = grid.predict(X_test)\n\n    print("Best params:", grid.best_params_)\n    print("Accuracy:", round(accuracy_score(y_test, pred), 4))\n    print(classification_report(y_test, pred, digits=4))\n\n\nif __name__ == "__main__":\n    main()\n'

Path("../scripts").mkdir(parents=True, exist_ok=True)
Path("../scripts/gridsearch_balanced_reviews.py").write_text(script, encoding="utf-8")
print("saved -> ../scripts/gridsearch_balanced_reviews.py")
